# ⚡ Feature Engineering for Power Consumption

## 📌 Introduction

Welcome to the Feature Engineering phase of the **Individual Household Electric Power Consumption** project! 🚀

> **💡 Why this step is important:**  
> Feature engineering is the process of using domain knowledge to extract new variables from raw data. Machine learning models, especially regression and time series models, cannot inherently understand raw timestamps or temporal dependencies. By creating meaningful features, we expose underlying patterns (like daily or weekly seasonality) to the model, significantly improving its predictive performance.

In [2]:
import numpy as np 
import pandas as pd

## 📂 Loading the Engineered Dataset

In this step, we load the pre-processed and cleaned dataset that was prepared in the previous stage.

> **🎯 Why this step is important:**  
> We need a clean, structured baseline to build upon. Loading the data from a highly optimized format like Parquet ensures faster read times and preserves data types, which is crucial for handling large time series datasets efficiently.

In [3]:
df = pd.read_parquet("cleaned_power.parquet")
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2006-12-16 17:24:00,4.216,0.418,234.839996,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.630005,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.289993,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.740005,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.679993,15.8,0.0,1.0,17.0


## 🔍 Dataset Inspection

Let's briefly inspect the shape and structure of our dataset to make sure everything looks correct. 🕵️‍♂️

> **🎯 Why this step is important:**  
> Before creating new features, it's essential to understand the current dimensions and data types of our DataFrame. This gives us a solid baseline to compare against after new features are added.

In [4]:
df.shape

(2075259, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2075259 entries, 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Data columns (total 7 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Global_active_power    float32
 1   Global_reactive_power  float32
 2   Voltage                float32
 3   Global_intensity       float32
 4   Sub_metering_1         float32
 5   Sub_metering_2         float32
 6   Sub_metering_3         float32
dtypes: float32(7)
memory usage: 71.2 MB


## 🎯 Target Variable

Before we extract new features, it's important to identify our target variable. In this project, our primary focus is predicting `Global_active_power`. 🔋

> **💡 Why this step is important:**  
> Understanding the target variable guides the feature engineering process. We need to create features that have a strong predictive relationship with the household's active power consumption.

## 📅 Date-Time Feature Extraction

Here, we extract specific components from the Datetime index, such as the hour, day, day of the week, month, and whether a day is a weekend. 📆

> **🏷️ What the feature represents:** Categorical and numerical representations of the timestamp.  
> **🧠 Why it is created:** Models cannot directly parse datetime objects. By explicitly breaking down the date and time, we allow the machine learning model to learn usage patterns specific to certain times.  
> **📈 How it helps regression models:** It helps the model capture human behavioral patterns, such as increased power usage on weekends or during specific evening hours.

In [6]:
df["hour"] = df.index.hour
df["day"] = df.index.day
df["day_of_week"] = df.index.dayofweek  # Monday=0, Sunday=6
df["month"] = df.index.month
df["quarter"] = df.index.quarter
df["dayofyear"] = df.index.dayofyear
df["weekofyear"] = df.index.isocalendar().week.astype(int)
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

## 🔄 Cyclical Encoding

We apply sine and cosine transformations to our cyclical time features (hour, day of week, and month). ⏱️

> **🏷️ What the feature represents:** A mathematical transformation that maps cyclical variables onto a circle, preserving their continuous nature.  
> **🧠 Why it is created:** In raw numerical form, hour 23 and hour 0 seem very far apart (a difference of 23), but they are only one hour apart! Cyclical encoding ensures that the transition from the end of a cycle back to the beginning is completely smooth.  
> **📈 How it helps regression models:** This allows the regression model to understand the true proximity of time points, preventing arbitrary jumps in predictions at the boundaries of days, weeks, or years.

In [7]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"]/24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"]/24)

df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"]/7)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"]/7)

df["month_sin"] = np.sin(2 * np.pi * df["month"]/12)
df["month_cos"] = np.cos(2 * np.pi * df["month"]/12)

## ⏪ Lag Features

We create lag features by shifting the `Global_active_power` variable backward in time (e.g., 1 minute, 60 minutes, or 1440 minutes ago). 🕰️

> **🏷️ What the feature represents:** The exact value of power consumption at a specific historical time step.  
> **🧠 Why it is created:** Time series data is highly autocorrelated, meaning past values are often excellent predictors of future values.  
> **📈 How it helps regression models:** By providing the model with historical context, it can directly leverage past behavior to make more accurate future forecasts.

In [8]:
df["lag_1"] = df["Global_active_power"].shift(1)
df["lag_60"] = df["Global_active_power"].shift(60)
df["lag_1440"] = df["Global_active_power"].shift(1440)

## 📊 Rolling Statistics

We calculate rolling averages and standard deviations of the `Global_active_power` over specific windows (e.g., past 60 minutes or past 24 hours). 📈

> **🏷️ What the feature represents:** The smoothed trend (mean) and volatility (standard deviation) of power consumption over a recent time window.  
> **🧠 Why it is created:** While lag features provide a single point in time, rolling statistics provide a broader summary of recent trends and stability.  
> **📈 How it helps regression models:** It helps the model understand the current state of the system—whether consumption is generally trending up, trending down, or experiencing high fluctuations, which makes predictions more robust to sudden spikes.

In [9]:
df["previous_hour_avg"] = df["Global_active_power"].shift(1).rolling(window=60).mean()
df["previous_day_avg"] = df["Global_active_power"].shift(1).rolling(window=1440).mean()

df["previous_hour_std"] = df["Global_active_power"].shift(1).rolling(window=60).std()
df["previous_day_std"] = df["Global_active_power"].shift(1).rolling(window=1440).std()

In [10]:
for col in df.iloc[:,-21:]:
    df[col] = df[col].astype("float32")

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2075259 entries, 2006-12-16 17:24:00 to 2010-11-26 21:02:00
Data columns (total 28 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Global_active_power    float32
 1   Global_reactive_power  float32
 2   Voltage                float32
 3   Global_intensity       float32
 4   Sub_metering_1         float32
 5   Sub_metering_2         float32
 6   Sub_metering_3         float32
 7   hour                   float32
 8   day                    float32
 9   day_of_week            float32
 10  month                  float32
 11  quarter                float32
 12  dayofyear              float32
 13  weekofyear             float32
 14  is_weekend             float32
 15  hour_sin               float32
 16  hour_cos               float32
 17  day_sin                float32
 18  day_cos                float32
 19  month_sin              float32
 20  month_cos              float32
 21  lag_1                

In [12]:
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,hour,day,day_of_week,...,day_cos,month_sin,month_cos,lag_1,lag_60,lag_1440,previous_hour_avg,previous_day_avg,previous_hour_std,previous_day_std
2006-12-16 17:24:00,4.216,0.418,234.839996,18.4,0.0,1.0,17.0,17.0,16.0,5.0,...,-0.222521,-2.449294e-16,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:25:00,5.360,0.436,233.630005,23.0,0.0,1.0,16.0,17.0,16.0,5.0,...,-0.222521,-2.449294e-16,1.0,4.216,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:26:00,5.374,0.498,233.289993,23.0,0.0,2.0,17.0,17.0,16.0,5.0,...,-0.222521,-2.449294e-16,1.0,5.360,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:27:00,5.388,0.502,233.740005,23.0,0.0,1.0,17.0,17.0,16.0,5.0,...,-0.222521,-2.449294e-16,1.0,5.374,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:28:00,3.666,0.528,235.679993,15.8,0.0,1.0,17.0,17.0,16.0,5.0,...,-0.222521,-2.449294e-16,1.0,5.388,NaN,NaN,NaN,NaN,NaN,NaN


## 🧹 Handling Missing Values

We check for missing values introduced by our feature engineering and drop them. 🗑️

> **💡 Why this step is important:**  
> Creating lag and rolling features naturally introduces `NaN` (Not a Number) values at the beginning of the dataset, because there is no historical data available for the very first few rows.  
>
> **📈 How it helps machine learning models:**  
> Most regression models cannot handle missing values out-of-the-box. Removing these initialization rows ensures our dataset is clean, complete, and ready for model training without causing errors.

In [13]:
df.isnull().sum()

Global_active_power         0
Global_reactive_power       0
Voltage                     0
Global_intensity            0
Sub_metering_1              0
Sub_metering_2              0
Sub_metering_3              0
hour                        0
day                         0
day_of_week                 0
month                       0
quarter                     0
dayofyear                   0
weekofyear                  0
is_weekend                  0
hour_sin                    0
hour_cos                    0
day_sin                     0
day_cos                     0
month_sin                   0
month_cos                   0
lag_1                       1
lag_60                     60
lag_1440                 1440
previous_hour_avg          60
previous_day_avg         1440
previous_hour_std          60
previous_day_std         1440
dtype: int64

In [14]:
df = df.dropna().copy()

## 💾 Saving the Feature Engineered Dataset

Finally, we save our fully processed and feature-rich dataset into a new Parquet file. 📦

> **🎯 Why this step is important:**  
> It creates a clear checkpoint in our data pipeline. By saving the result, we can easily load this finalized dataset into our machine learning notebooks without having to recompute all the features from scratch every time!

In [15]:
df.to_parquet("featured_power.parquet")

---

## 🏆 Summary / Key Takeaways

We have successfully transformed our raw time series data into a rich dataset ready for machine learning! 🎉

### ✨ Features Created:
- **📅 Date-Time Features:** Hour, day, month, weekend flags.
- **🔄 Cyclical Features:** Sine and cosine transformations.
- **⏪ Lag Features:** Historical values from past time steps.
- **📊 Rolling Statistics:** Moving averages and standard deviations.

### 🧠 Why & How They Help:
These features bridge the gap between raw timestamps and machine learning algorithms. By explicitly encoding time patterns, ensuring cyclical continuity, and providing historical context, we empower regression models to capture complex temporal patterns and seasonality!

### 🚀 Conclusion
*With missing values handled and our new features optimized and saved, this engineered dataset is now fully prepped and ready for the regression modeling phase!* 🥳